# 画像生成アプリケーションの構築（OpenAI）

LLM（大規模言語モデル）はテキスト生成だけではありません。テキストの説明から画像を生成することもできます。画像というモダリティは、医療技術、建築、観光、ゲーム開発、マーケティングなど幅広い分野で役立ちます。このレッスンでは、OpenAIプラットフォーム上の最新の<strong>GPT Image</strong>モデルを使っていきます。

## 学習目標

このレッスンの終わりまでに以下ができるようになります：

- 画像生成とは何か、その利用が有用な場面を説明できる。
- `gpt-image`モデルファミリーを理解し、従来のDALL·Eモデルとどう異なるか説明できる。
- 画像生成アプリケーションを構築し、画像編集ができる。

## 画像生成とは？

画像生成モデルはテキストプロンプトから画像を作成します。`gpt-image`のような最新モデルは、訓練中にテキストと画像の関係を学習し、ランダムノイズを反復処理してプロンプトに合う画像に変換します。

よく知られた画像モデルのファミリーは以下の2つです：

- **`gpt-image`（OpenAI）** — 本レッスンで使う最新世代。テキストから画像生成や、マスクを使った画像編集（インペインティング）に対応。
- **Midjourney** — 独自サービスとDiscordベースのワークフローを持つ人気のサードパーティモデル。

> 以前のOpenAI画像モデルである<strong>DALL·E 2</strong>と<strong>DALL·E 3</strong>はレガシーです。DALL·E 3は新規デプロイには利用できず、`create_variation`などの機能はDALL·E 2でのみ存在しました。新規アプリケーションには`gpt-image`モデルを使いましょう。

> **重要：** `gpt-image`モデルは生成した画像を<strong>base64</strong>形式（`b64_json`）で返します。画像URLは返しません。コードはbase64文字列をデコードしてバイトデータに変換し保存します。画像をダウンロードするURLはありません。


## はじめての画像生成アプリケーションの構築

画像生成アプリケーションを構築するには何が必要でしょうか？以下のライブラリが必要です：

- **python-dotenv**、秘密情報をコードから分離して *.env* ファイルで管理するためにこのライブラリの使用を強く推奨します。
- **openai**、OpenAIのAPIとやり取りするために使用するライブラリです。
- **pillow**、Pythonで画像を扱うためのライブラリです。


1. 次の内容で *.env* ファイルを作成します：

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. 上記のライブラリを *requirements.txt* というファイルに次のようにまとめます：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 次に、仮想環境を作成してライブラリをインストールします：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsの場合は、以下のコマンドを使って仮想環境を作成し、有効化してください:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* というファイルに以下のコードを追加します:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # dotenvをインポートする
    dotenv.load_dotenv()

    # OpenAIオブジェクトを作成する（.envからOPENAI_API_KEYを読み込みます）
    client = openai.OpenAI()


    try:
        # 画像生成APIを使用して画像を作成する
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ここにプロンプトテキストを入力してください
            size='1024x1024',
            n=1
        )
        # 保存する画像のディレクトリを設定する
        image_dir = os.path.join(os.curdir, 'images')

        # ディレクトリが存在しない場合は作成する
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 画像パスを初期化する（ファイルタイプはpngにしてください）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-imageモデルは画像をURLではなくbase64（b64_json）で返します
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # デフォルトの画像ビューアで画像を表示する
        image = Image.open(image_path)
        image.show()

    # 例外をキャッチする
    except openai.BadRequestError as err:
        print(err)

    ```

このコードについて説明しましょう:

- まず、必要なライブラリをインポートします。OpenAIライブラリ、dotenvライブラリ、base64モジュール、Pillowライブラリを含みます。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- その後、クライアントを作成します。これは ``.env`` ファイルからAPIキーを読み込みます。

    ```python
    # OpenAIオブジェクトを作成する
    client = openai.OpenAI()
    ```

- 次に、画像を生成します:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ここにプロンプトテキストを入力してください
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` モデルは画像を `data[0].b64_json` に<strong>base64</strong>文字列として返します。これをバイトにデコードしてファイルに書き込みます — ダウンロード用のURLはありません。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後に、画像を開いて標準の画像ビューアで表示します:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 画像生成の詳細

`images.generate` のパラメーターを見てみましょう:

- **model** は使用する画像モデルです（例: `gpt-image-1`）。
- **prompt** は画像生成に使うテキストプロンプトです。ここでは「Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils」です。
- **size** は生成される画像のサイズです（例: `1024x1024`, `1536x1024`, `1024x1536`, または `"auto"`）。
- **n** は生成する画像の数です。ここでは1枚生成しています。

> 画像モデルは `temperature` パラメーターを取りません — それはテキスト生成の制御です。多様性を得るには再度APIを呼び出し、少なくしたい場合はより具体的なプロンプトにしてください。

## 画像生成の追加機能

Pythonの数行で画像を生成する方法を見てきました。`gpt-image` モデルは既存の画像を<strong>編集</strong>することもできます。画像、オプションの<strong>マスク</strong>（変更する部分を示す）、そしてプロンプトを指定すると、画像の一部を変更できます — 例えば、ウサギに帽子をかぶせることができます。

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編集はbase64形式でも返されます
edited_image = base64.b64decode(response.data[0].b64_json)
```

元の画像はウサギだけですが、最終画像ではウサギに帽子が追加されています。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
